In [0]:
df = spark.table("workspace.default.churn_features")
display(df.limit(5))

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
feature_columns = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary",
    "Geography_Index",
    "Gender_Index"
]

label_column = "Exited"

In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

df_with_features = assembler.transform(df)

final_predictions = rf_model.transform(df_with_features)

In [0]:
# train_df, test_df = df_model.randomSplit([0.8, 0.2], seed=42)
train_df, test_df = df_with_features.randomSplit([0.8, 0.2], seed=42)

In [0]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol=label_column,
    numTrees=100,
    maxDepth=8,
    seed=42
)

rf_model = rf.fit(train_df)

In [0]:
rf_predictions = rf_model.transform(test_df)

display(rf_predictions.select("Exited", "prediction", "probability").limit(10))

Exited,prediction,probability
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.7999206456887081"",""0.2000793543112919""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9714539469380468"",""0.028546053061953267""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9420468865343103"",""0.057953113465689654""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.7796602426882958"",""0.2203397573117043""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9652792460814327"",""0.03472075391856739""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9519071502360552"",""0.04809284976394475""]}"
1,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.8804471874875561"",""0.11955281251244382""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9190076865028064"",""0.08099231349719353""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9128753765632796"",""0.08712462343672048""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.8828061817776771"",""0.11719381822232298""]}"


In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

def extract_prob(v):
    return float(v[1])

extract_udf = udf(extract_prob, DoubleType())

rf_predictions = rf_predictions.withColumn(
    "churn_probability",
    extract_udf("probability")
)

display(rf_predictions.select("Exited", "churn_probability").limit(10))

Exited,churn_probability
0,0.2000793543112919
0,0.028546053061953267
0,0.057953113465689654
0,0.2203397573117043
0,0.03472075391856739
0,0.04809284976394475
1,0.11955281251244382
0,0.08099231349719353
0,0.08712462343672048
0,0.11719381822232298


In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

rf_evaluator = BinaryClassificationEvaluator(
    labelCol="Exited",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

rf_auc = rf_evaluator.evaluate(rf_predictions)
print("Random Forest AUC:", rf_auc)

Random Forest AUC: 0.8635413388481769


In [0]:
from pyspark.sql.functions import col, when, sum as _sum

rf_metrics_df = rf_predictions.select(
    col("Exited").alias("label"),
    col("prediction")
)

rf_cm = rf_metrics_df.select(
    _sum(when((col("label") == 1) & (col("prediction") == 1), 1).otherwise(0)).alias("TP"),
    _sum(when((col("label") == 0) & (col("prediction") == 0), 1).otherwise(0)).alias("TN"),
    _sum(when((col("label") == 0) & (col("prediction") == 1), 1).otherwise(0)).alias("FP"),
    _sum(when((col("label") == 1) & (col("prediction") == 0), 1).otherwise(0)).alias("FN")
)

rf_cm.show()

+---+----+---+---+
| TP|  TN| FP| FN|
+---+----+---+---+
|171|1470| 38|242|
+---+----+---+---+



In [0]:
rf_row = rf_cm.collect()[0]

TP = rf_row["TP"]
TN = rf_row["TN"]
FP = rf_row["FP"]
FN = rf_row["FN"]

rf_precision = TP / (TP + FP) if (TP + FP) != 0 else 0
rf_recall = TP / (TP + FN) if (TP + FN) != 0 else 0
rf_f1 = 2 * (rf_precision * rf_recall) / (rf_precision + rf_recall) if (rf_precision + rf_recall) != 0 else 0
rf_accuracy = (TP + TN) / (TP + TN + FP + FN)

print("Random Forest Accuracy:", rf_accuracy)
print("Random Forest Precision:", rf_precision)
print("Random Forest Recall:", rf_recall)
print("Random Forest F1 Score:", rf_f1)

Random Forest Accuracy: 0.8542425819885476
Random Forest Precision: 0.8181818181818182
Random Forest Recall: 0.41404358353510895
Random Forest F1 Score: 0.5498392282958199


In [0]:
from pyspark.sql.functions import when, col

threshold = 0.3

rf_predictions_custom = rf_predictions.withColumn(
    "custom_prediction",
    when(col("churn_probability") >= threshold, 1).otherwise(0)
)

display(
    rf_predictions_custom.select("Exited", "churn_probability", "custom_prediction").limit(10)
)

Exited,churn_probability,custom_prediction
0,0.2000793543112919,0
0,0.028546053061953267,0
0,0.057953113465689654,0
0,0.2203397573117043,0
0,0.03472075391856739,0
0,0.04809284976394475,0
1,0.11955281251244382,0
0,0.08099231349719353,0
0,0.08712462343672048,0
0,0.11719381822232298,0


In [0]:
rf_metrics_custom_df = rf_predictions_custom.select(
    col("Exited").alias("label"),
    col("custom_prediction").alias("prediction")
)

rf_cm_custom = rf_metrics_custom_df.select(
    _sum(when((col("label") == 1) & (col("prediction") == 1), 1).otherwise(0)).alias("TP"),
    _sum(when((col("label") == 0) & (col("prediction") == 0), 1).otherwise(0)).alias("TN"),
    _sum(when((col("label") == 0) & (col("prediction") == 1), 1).otherwise(0)).alias("FP"),
    _sum(when((col("label") == 1) & (col("prediction") == 0), 1).otherwise(0)).alias("FN")
)

rf_cm_custom.show()

+---+----+---+---+
| TP|  TN| FP| FN|
+---+----+---+---+
|258|1390|118|155|
+---+----+---+---+



In [0]:
rf_row_custom = rf_cm_custom.collect()[0]

TP = rf_row_custom["TP"]
TN = rf_row_custom["TN"]
FP = rf_row_custom["FP"]
FN = rf_row_custom["FN"]

rf_precision_custom = TP / (TP + FP) if (TP + FP) != 0 else 0
rf_recall_custom = TP / (TP + FN) if (TP + FN) != 0 else 0
rf_f1_custom = 2 * (rf_precision_custom * rf_recall_custom) / (rf_precision_custom + rf_recall_custom) if (rf_precision_custom + rf_recall_custom) != 0 else 0
rf_accuracy_custom = (TP + TN) / (TP + TN + FP + FN)

print("RF Custom Accuracy:", rf_accuracy_custom)
print("RF Custom Precision:", rf_precision_custom)
print("RF Custom Recall:", rf_recall_custom)
print("RF Custom F1 Score:", rf_f1_custom)

RF Custom Accuracy: 0.8578865174388339
RF Custom Precision: 0.6861702127659575
RF Custom Recall: 0.6246973365617433
RF Custom F1 Score: 0.6539923954372624


In [0]:
rf_importance = list(zip(feature_columns, rf_model.featureImportances))

rf_importance_sorted = sorted(rf_importance, key=lambda x: x[1], reverse=True)

for feature, importance in rf_importance_sorted:
    print(feature, ":", importance)

Age : 0.3764640685907541
NumOfProducts : 0.28005033681246305
IsActiveMember : 0.09693051423425571
Balance : 0.0657406505593753
Geography_Index : 0.051312518170453555
CreditScore : 0.0412830901804984
EstimatedSalary : 0.03875828947671016
Tenure : 0.029485538252725974
Gender_Index : 0.014684933706717761
HasCrCard : 0.0052900600160461116


In [0]:
rf_high_risk = (
    rf_predictions
    .select("CustomerId", "churn_probability", "Balance", "IsActiveMember")
    .orderBy("churn_probability", ascending=False)
    .limit(100)
)

display(rf_high_risk)

CustomerId,churn_probability,Balance,IsActiveMember
15692416,0.9793030866963002,143542.36,0
15640846,0.9673592643785599,106458.31,0
15704442,0.9656275840849214,169406.33,1
15783019,0.9609255858171336,123681.32,0
15734762,0.9599602003255504,115895.22,0
15800061,0.9586028345674243,89158.94,0
15731815,0.9582012861141429,96134.11,0
15599329,0.955329486274422,195238.29,1
15786196,0.9537910066549448,105770.7,0
15673591,0.9493521132899493,141252.18,1


In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import col
from pyspark.ml.functions import vector_to_array

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

df_with_features = assembler.transform(df)

final_predictions = rf_model.transform(df_with_features)

final_predictions = final_predictions.withColumn(
    "probability_array",
    vector_to_array(col("probability"))
)

final_predictions = final_predictions.withColumn(
    "churn_probability",
    col("probability_array")[1]
)

display(
    final_predictions.select("CustomerId", "churn_probability").limit(10)
)

CustomerId,churn_probability
15634602,0.3259683792041926
15647311,0.13650746059749314
15619304,0.9598618842367589
15701354,0.07886992024886977
15737888,0.22673346923310306
15574012,0.1558637129904258
15592531,0.09064098243848147
15656148,0.7888202383007837
15792365,0.1609618138451211
15592389,0.07425438409394709


In [0]:
final_predictions.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.churn_predictions")

In [0]:
pred_df = spark.table("workspace.default.churn_predictions")
display(pred_df.select("CustomerId", "churn_probability").limit(10))

CustomerId,churn_probability
15634602,0.3259683792041926
15647311,0.13650746059749314
15619304,0.9598618842367589
15701354,0.07886992024886977
15737888,0.22673346923310306
15574012,0.1558637129904258
15592531,0.09064098243848147
15656148,0.7888202383007837
15792365,0.1609618138451211
15592389,0.07425438409394709


In [0]:
from pyspark.sql.functions import when, col

threshold = 0.3

final_predictions = final_predictions.withColumn(
    "risk_flag",
    when(col("churn_probability") >= threshold, 1).otherwise(0)
)

high_risk_customers = final_predictions.filter(col("risk_flag") == 1)

In [0]:
display(
    high_risk_customers
    .select(
        "CustomerId",
        "churn_probability",
        "Balance",
        "IsActiveMember",
        "Age",
        "NumOfProducts",
        "Geography"
    )
    .orderBy("churn_probability", ascending=False)
    .limit(100)
)

CustomerId,churn_probability,Balance,IsActiveMember,Age,NumOfProducts,Geography
15648681,0.9919609852102689,139914.6,1,47,4,France
15666548,0.983237761355318,111920.13,0,56,3,Germany
15570002,0.9820316877153223,118772.71,0,55,4,Germany
15752342,0.9807624142316569,133656.91,0,54,3,Germany
15692416,0.9793030866963002,143542.36,0,52,3,Spain
15696989,0.9791587759839235,139493.25,0,52,3,Germany
15775318,0.9790403362483899,154962.99,1,51,3,Spain
15671591,0.9788764069184954,96196.24,0,52,4,Spain
15787189,0.9781452077105616,134250.17,0,60,3,Germany
15679770,0.9775568770142143,131583.59,1,61,4,France


In [0]:
high_risk_customers.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.high_risk_customers")

In [0]:
risk_df = spark.table("workspace.default.high_risk_customers")

display(
    risk_df.select(
        "CustomerId",
        "churn_probability",
        "Balance",
        "IsActiveMember"
    ).orderBy("churn_probability", ascending=False).limit(10)
)

CustomerId,churn_probability,Balance,IsActiveMember
15648681,0.9919609852102689,139914.6,1
15666548,0.983237761355318,111920.13,0
15570002,0.9820316877153223,118772.71,0
15752342,0.9807624142316569,133656.91,0
15692416,0.9793030866963002,143542.36,0
15696989,0.9791587759839235,139493.25,0
15775318,0.9790403362483899,154962.99,1
15671591,0.9788764069184954,96196.24,0
15787189,0.9781452077105616,134250.17,0
15679770,0.9775568770142143,131583.59,1


# MLflow Tracking


In [0]:
import mlflow
import mlflow.spark

In [0]:
mlflow.set_experiment("/Users/patelkaran3639@gmail.com/Customer_Churn_RF")

<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/2813043112847102', creation_time=1776931709085, experiment_id='2813043112847102', last_update_time=1776932011362, lifecycle_stage='active', name='/Users/patelkaran3639@gmail.com/Customer_Churn_RF', tags={'mlflow.experiment.sourceName': '/Users/patelkaran3639@gmail.com/Customer_Churn_RF',
 'mlflow.experimentKind': 'custom_model_development',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'patelkaran3639@gmail.com',
 'mlflow.ownerId': '77639191990804'}>

In [0]:
import os
os.environ["MLFLOW_DFS_TMP"] = "/tmp/mlflow"
print("MLFLOW_DFS_TMP set")

MLFLOW_DFS_TMP set


In [0]:
mlflow_tmp_path = "/Volumes/workspace/default/churn_files/mlflow_tmp"
print(mlflow_tmp_path)

/Volumes/workspace/default/churn_files/mlflow_tmp


In [0]:
with mlflow.start_run(run_name="random_forest_churn_model"):

    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("numTrees", 100)
    mlflow.log_param("maxDepth", 8)
    mlflow.log_param("threshold", 0.3)

    mlflow.log_metric("auc", rf_auc)
    mlflow.log_metric("accuracy_default", rf_accuracy)
    mlflow.log_metric("precision_default", rf_precision)
    mlflow.log_metric("recall_default", rf_recall)
    mlflow.log_metric("f1_default", rf_f1)

    mlflow.log_metric("accuracy_custom", rf_accuracy_custom)
    mlflow.log_metric("precision_custom", rf_precision_custom)
    mlflow.log_metric("recall_custom", rf_recall_custom)
    mlflow.log_metric("f1_custom", rf_f1_custom)

    mlflow.spark.log_model(
    rf_model,
    "random_forest_model",
    dfs_tmpdir=mlflow_tmp_path
)

2026/04/23 20:15:15 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.1.0+databricks.connect.18.0.2) contains a local version label (+databricks.connect.18.0.2). MLflow logged a pip requirement for this package as 'pyspark==4.1.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/04/23 20:15:19 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-a1098888-c42f-457a-a51f-15/tmpuy98bpc7/model, flavor: spark). Fall back to return ['pyspark==4.1.0']. Set logging level to DEBUG to see the full traceback. 
2026/04/23 20:15:19 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when l

Scoring Full Dataset


In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import col, when
from pyspark.ml.functions import vector_to_array

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

df_with_features = assembler.transform(df)

final_predictions = rf_model.transform(df_with_features)

final_predictions = final_predictions.withColumn(
    "probability_array",
    vector_to_array(col("probability"))
)

final_predictions = final_predictions.withColumn(
    "churn_probability",
    col("probability_array")[1]
)

threshold = 0.3

final_predictions = final_predictions.withColumn(
    "risk_flag",
    when(col("churn_probability") >= threshold, 1).otherwise(0)
)

display(
    final_predictions.select(
        "CustomerId",
        "churn_probability",
        "risk_flag",
        "Balance",
        "IsActiveMember",
        "Geography"
    ).limit(10)
)

CustomerId,churn_probability,risk_flag,Balance,IsActiveMember,Geography
15634602,0.3259683792041926,1,0.0,1,France
15647311,0.13650746059749314,0,83807.86,1,Spain
15619304,0.9598618842367589,1,159660.8,0,France
15701354,0.07886992024886977,0,0.0,0,France
15737888,0.22673346923310306,0,125510.82,1,Spain
15574012,0.1558637129904258,0,113755.78,0,Spain
15592531,0.09064098243848147,0,0.0,1,France
15656148,0.7888202383007837,1,115046.74,0,Germany
15792365,0.1609618138451211,0,142051.07,1,France
15592389,0.07425438409394709,0,134603.88,1,France


In [0]:
final_predictions.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.churn_predictions")

In [0]:
high_risk_customers = final_predictions.filter(col("risk_flag") == 1)

high_risk_customers.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.high_risk_customers")